# Scenario Analysis

This notebook answers the core improvement question:

> **How much does co-scheduling multiple cancer screenings into fewer patient
> encounters improve screening rates, reduce loss-to-follow-up, and increase
> detected cancers?**

## Scenarios
| Scenario | Description |
|---|---|
| `baseline_fragmented` | Current state — each provider screens their domain only; separate appointments per cancer |
| `gyn_coordinated` | GYN visits bundle cervical + breast; rest fragmented |
| `coordinated_all` | All due screenings bundled into one encounter per patient |
| `high_access_coordinated` | Full co-scheduling + dramatically reduced scheduling friction |

## Key Mechanic
- **Fragmented**: each cancer = one separate encounter, one independent LTFU risk.
- **Coordinated**: all due cancers = one encounter, one shared LTFU risk → fewer drop-out points.

All scenarios run on the **same patient cohort** so results are directly comparable.

In [1]:
import sys, random
sys.path.insert(0, '../src')

import config as cfg
from population import sample_patient
from scenarios import (
    SCENARIOS,
    compare_scenarios,
    get_multi_screening_eligible,
    age_cluster_summary,
)
from metrics import compute_rates, print_summary

random.seed(cfg.RANDOM_SEED)
print('Imports OK')
print(f'Scenarios available: {list(SCENARIOS.keys())}')

Imports OK
Scenarios available: ['baseline_fragmented', 'gyn_coordinated', 'coordinated_all', 'high_access_coordinated']


## Generate Shared Patient Cohort

All scenarios run on the same patients so comparisons are apples-to-apples.

In [2]:
from collections import Counter

N_PATIENTS = 2_000
SIM_DAY    = 0

# Provider assignment mirrors Sophia's destination probabilities
PROVIDER_PROBS = list(cfg.DESTINATION_PROBS.items())
providers_pool = [p for p, _ in PROVIDER_PROBS]
provider_weights = [w for _, w in PROVIDER_PROBS]

random.seed(cfg.RANDOM_SEED)
providers = random.choices(providers_pool, weights=provider_weights, k=N_PATIENTS)
patients  = [
    sample_patient(i, SIM_DAY, providers[i], 'outpatient')
    for i in range(N_PATIENTS)
]

print(f'Cohort: {N_PATIENTS:,} patients')
print(f'Age range: {min(p.age for p in patients)}–{max(p.age for p in patients)}')
print(f'Provider mix: {dict(Counter(providers))}')
print(f'Has cervix: {sum(p.has_cervix for p in patients):,}')
print(f'Smokers:    {sum(p.smoker for p in patients):,}')
print(f'HPV+:       {sum(p.hpv_positive for p in patients):,}')

Cohort: 2,000 patients
Age range: 21–80
Provider mix: {'specialist': 421, 'pcp': 697, 'er': 399, 'gynecologist': 483}
Has cervix: 1,844
Smokers:    263
HPV+:       374


## Age-Clustering Analysis

Identify where multiple screening opportunities converge — the co-scheduling sweet spot.
Clinical hypothesis: women aged ~40–50 are simultaneously due for cervical, breast, and colorectal screening.

In [ ]:
from screening import get_eligible_screenings

cluster = age_cluster_summary(patients)
multi   = get_multi_screening_eligible(patients)

print(f'Patients eligible for ≥2 screenings: {len(multi):,} ({100*len(multi)/N_PATIENTS:.1f}%)')
print(f'  → prime co-scheduling candidates\n')

print(f'{"Age group":<12}  1 screening  2 screenings  3+ screenings  Total')
print('-' * 65)
for age_grp, counts in sorted(cluster.items()):
    one   = counts.get('1_screenings', 0)
    two   = counts.get('2_screenings', 0)
    three = sum(v for k, v in counts.items() if k not in ('0_screenings','1_screenings','2_screenings'))
    total = sum(counts.values())
    print(f'{age_grp:<12}  {one:>11}  {two:>12}  {three:>13}  {total:>5}')

## Run All Scenarios

Each scenario runs on a **deep copy** of the same cohort — results are independent.

In [ ]:
all_scenario_names = list(SCENARIOS.keys())

print(f'Running {len(all_scenario_names)} scenarios on {N_PATIENTS:,} patients...\n')
results = compare_scenarios(all_scenario_names, patients, providers, SIM_DAY, seed=42)

for name, m in results.items():
    print(f"  ✓ {SCENARIOS[name]['label']}  — screened cervical: {m['n_screened']['cervical']:,}")

## Side-by-Side Comparison Table

In [ ]:
rate_labels = {
    'screening_rate_cervical_pct':  'Cervical screening rate',
    'unscreened_pct':               'Unscreened rate',
    'abnormal_rate_cervical_pct':   'Abnormal Pap rate (of screened)',
    'colposcopy_completion_pct':    'Colposcopy completion',
    'treatment_completion_pct':     'Treatment completion',
    'ltfu_rate_pct':                'Overall LTFU rate',
}

# Header
labels_col = [SCENARIOS[n]['label'] for n in all_scenario_names]
col_w = 22
header = f'{"Metric":<42}' + ''.join(f'{l[:col_w]:>{col_w}}' for l in labels_col)
print(header)
print('-' * (42 + col_w * len(all_scenario_names)))

rates_by_scenario = {name: compute_rates(results[name]) for name in all_scenario_names}

for key, label in rate_labels.items():
    row = f'{label:<42}'
    for name in all_scenario_names:
        val = rates_by_scenario[name][key]
        row += f'{val:>{col_w}.1f}%'
    print(row)

# Encounters saved (co-scheduling efficiency)
print()
enc_row = f'{"Encounters saved (vs. fragmented)":<42}'
baseline_enc = results['baseline_fragmented'].get('scenario_encounters_saved', 0)
for name in all_scenario_names:
    saved = results[name].get('scenario_encounters_saved', 0)
    enc_row += f'{saved:>{col_w},}'
print(enc_row)

## Absolute Improvement Over Baseline

Delta vs. `baseline_fragmented` for each metric — the direct case for co-scheduling.

In [ ]:
baseline_rates = rates_by_scenario['baseline_fragmented']
improvement_scenarios = [n for n in all_scenario_names if n != 'baseline_fragmented']

labels_col_imp = [SCENARIOS[n]['label'] for n in improvement_scenarios]
header_imp = f'{"Metric (Δ vs. baseline)":<42}' + ''.join(f'{l[:col_w]:>{col_w}}' for l in labels_col_imp)
print(header_imp)
print('-' * (42 + col_w * len(improvement_scenarios)))

for key, label in rate_labels.items():
    row = f'{label:<42}'
    baseline_val = baseline_rates[key]
    for name in improvement_scenarios:
        delta = rates_by_scenario[name][key] - baseline_val
        sign  = '+' if delta >= 0 else ''
        row  += f'{sign}{delta:>{col_w-1}.1f}%'
    print(row)

## Cervical Pathway Funnel — All Scenarios

How many patients make it through each stage of the cervical pathway per scenario.

In [ ]:
funnel_steps = [
    ('Patients in cohort',          lambda m: m['n_patients']),
    ('Eligible for ≥1 screening',   lambda m: m['n_eligible_any']),
    ('Cervical screenings done',    lambda m: m['n_screened']['cervical']),
    ('Abnormal results',            lambda m: sum(
        v for k, v in m['cervical_results'].items() if k != 'NORMAL'
    )),
    ('Colposcopies completed',      lambda m: m['n_colposcopy']),
    ('Treated (LEEP/cone)',         lambda m: m['n_treatment'].get('leep', 0)
                                             + m['n_treatment'].get('cone_biopsy', 0)),
]

print(f'{"Stage":<36}' + ''.join(f'{SCENARIOS[n]["label"][:col_w]:>{col_w}}' for n in all_scenario_names))
print('-' * (36 + col_w * len(all_scenario_names)))

for step_label, extractor in funnel_steps:
    row = f'{step_label:<36}'
    for name in all_scenario_names:
        val = extractor(results[name])
        row += f'{val:>{col_w},}'
    print(row)

## Full Scenario Reports

Individual `print_summary` for each scenario — detailed breakdown.

In [ ]:
for name in all_scenario_names:
    print(f'\n{"="*65}')
    print(f'SCENARIO: {SCENARIOS[name]["label"]}')
    print(f'{SCENARIOS[name]["description"]}')
    print(f'{"="*65}')
    print_summary(results[name])

## Co-Scheduling Efficiency — Encounter Savings

How many separate appointments are eliminated by bundling screenings?
Each saved encounter = reduced scheduling friction + reduced LTFU risk.

In [ ]:
print('Encounter savings by scenario (vs. fully fragmented):')
print('-' * 55)
baseline_saved = results['baseline_fragmented'].get('scenario_encounters_saved', 0)

for name in all_scenario_names:
    saved  = results[name].get('scenario_encounters_saved', 0)
    label  = SCENARIOS[name]['label']
    pct    = 100 * saved / max(N_PATIENTS, 1)
    print(f'  {label:<35} {saved:>6,} encounters saved  ({pct:.1f}% of cohort)')

## Plots

Visual comparison of key rates across scenarios.

In [ ]:
try:
    import matplotlib.pyplot as plt
    import numpy as np

    scenario_labels = [SCENARIOS[n]['label'] for n in all_scenario_names]
    x = np.arange(len(all_scenario_names))
    width = 0.2

    metrics_to_plot = {
        'Cervical screening rate (%)':     'screening_rate_cervical_pct',
        'LTFU rate (%)':                   'ltfu_rate_pct',
        'Colposcopy completion (%)':       'colposcopy_completion_pct',
    }

    fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(14, 5))
    colors = ['#c0392b', '#e67e22', '#2980b9', '#27ae60']

    for ax, (title, key) in zip(axes, metrics_to_plot.items()):
        vals = [rates_by_scenario[n][key] for n in all_scenario_names]
        bars = ax.bar(x, vals, color=colors, edgecolor='white', width=0.6)
        ax.set_title(title, fontsize=11)
        ax.set_xticks(x)
        ax.set_xticklabels([SCENARIOS[n]['label'] for n in all_scenario_names],
                           rotation=15, ha='right', fontsize=8)
        ax.set_ylabel('%')
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    f'{val:.1f}%', ha='center', va='bottom', fontsize=8)

    fig.suptitle('Scenario Comparison — Co-Scheduling Impact', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

except ImportError:
    print('matplotlib not available — install with: pip install matplotlib')

## Notes for Clinical / OR Team

**What to calibrate before using these results:**
1. `ltfu_multiplier` per scenario — currently estimated; should come from literature on
   care coordination programs or NYP pilot data.
2. `PROVIDER_CANCER_MAP` in `scenarios.py` — verify which cancers each provider type
   currently screens for at NYP.
3. `scheduling_delay_days` per scenario — model with real NYP scheduling lead-times.
4. `capacity_multiplier` — coordinated programs require more screening slots per encounter;
   needs actual capacity data from each service line.

**Adding a new scenario:**
```python
# In scenarios.py, add to the SCENARIOS dict:
"my_scenario": {
    "label":                 "My New Scenario",
    "description":           "...",
    "cancer_map":            COORDINATED_CANCER_MAP,   # or custom
    "co_schedule":           True,
    "ltfu_multiplier":       0.65,
    "scheduling_delay_days": 14,
    "capacity_multiplier":   1.15,
}
```
Then re-run this notebook — it picks up new scenarios automatically.